# Embedding Model Evaluation

This notebook evaluates the `BAAI/bge-m3` sentence embedding model on a small set of English and Arabic texts.

In [10]:
#%pip install numpy
%pip install sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
from embedding_utils import load_embedding_model, embed_texts, compute_cosine_similarity

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Embedding Model

We load the `BAAI/bge-m3` sentence transformer model.

In [12]:
model = load_embedding_model("BAAI/bge-m3")
model

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14765.09it/s]


SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

## 2. Texts

Define the sample texts used for embedding evaluation.

In [13]:
texts = [
    "Saudi Aramco reports 12% increase in quarterly profit",
    "Aramco posts strong Q1 earnings with 12 percent profit growth",
    "أرامكو تعلن ارتفاع أرباحها الفصلية بنسبة 12٪",
    "Oil prices fall amid global recession concerns"
]
texts

['Saudi Aramco reports 12% increase in quarterly profit',
 'Aramco posts strong Q1 earnings with 12 percent profit growth',
 'أرامكو تعلن ارتفاع أرباحها الفصلية بنسبة 12٪',
 'Oil prices fall amid global recession concerns']

## 3. Generate Embeddings

Encode the sample texts into vector embeddings.

In [14]:
embeddings = embed_texts(model, texts, normalize_embeddings=True)
embeddings.shape

(4, 1024)

## 4. Compute Cosine Similarity

Compute pairwise cosine similarity between the generated embeddings.

In [15]:
similarity = compute_cosine_similarity(embeddings)
print(np.round(similarity, 3))

[[1.    0.798 0.785 0.327]
 [0.798 1.    0.758 0.355]
 [0.785 0.758 1.    0.345]
 [0.327 0.355 0.345 1.   ]]


## 5. Interpret Results

- Similar sentences should show high similarity values.
- Unrelated sentences should show low similarity values.
- Cross-language sentence pairs should show reasonable similarity if the model handles Arabic and English well.

6. OpenAI Embedding Model

In [16]:
%pip install openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Install OpenAI if needed
# %pip install openai

from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

openai_model = "text-embedding-3-small"

In [18]:
response = client.embeddings.create(
    model=openai_model,
    input=texts
)

openai_embeddings = np.array([item.embedding for item in response.data])

openai_embeddings.shape

(4, 1536)

In [19]:
def cosine_similarity_matrix(vectors):
    norm = np.linalg.norm(vectors, axis=1, keepdims=True)
    normalized = vectors / norm
    return np.dot(normalized, normalized.T)

openai_similarity = cosine_similarity_matrix(openai_embeddings)

print(np.round(openai_similarity, 3))

[[1.    0.88  0.526 0.343]
 [0.88  1.    0.519 0.333]
 [0.526 0.519 1.    0.202]
 [0.343 0.333 0.202 1.   ]]


In [20]:
print("BGE-m3 Similarity:\n")
print(np.round(similarity, 3))

print("\nOpenAI text-embedding-3-small Similarity:\n")
print(np.round(openai_similarity, 3))

BGE-m3 Similarity:

[[1.    0.798 0.785 0.327]
 [0.798 1.    0.758 0.355]
 [0.785 0.758 1.    0.345]
 [0.327 0.355 0.345 1.   ]]

OpenAI text-embedding-3-small Similarity:

[[1.    0.88  0.526 0.343]
 [0.88  1.    0.519 0.333]
 [0.526 0.519 1.    0.202]
 [0.343 0.333 0.202 1.   ]]
